# Transportation Problem — Formulation and Solution (Python)

> **ORKit Free** · Linear Programming (LP)

---

## What is the Transportation Problem?

A set of **suppliers** (factories, warehouses) must ship goods to a set of **customers** (stores, cities).
Each supplier has a limited supply, each customer has a demand to be met, and shipping from
supplier $i$ to customer $j$ has a known unit cost $c_{ij}$.

The goal: **minimize total transportation cost** while satisfying all demands and respecting supplies.

This is one of the earliest LP problems ever studied (Hitchcock, 1941; Koopmans, 1949) and
appears everywhere:
- Supply chain logistics
- Distribution network design
- Warehouse-to-store allocation
- Resource distribution in humanitarian operations

---

## Mathematical Formulation

### Sets and Parameters

| Symbol | Description |
|--------|-------------|
| $I$ | Set of suppliers ($m$ total) |
| $J$ | Set of customers ($n$ total) |
| $s_i$ | Supply available at supplier $i$ |
| $d_j$ | Demand required by customer $j$ |
| $c_{ij}$ | Unit shipping cost from $i$ to $j$ |

### Decision Variables

$$x_{ij} \geq 0: \text{quantity shipped from supplier } i \text{ to customer } j$$

### Formulation

$$\min \quad \sum_{i \in I} \sum_{j \in J} c_{ij} \, x_{ij}$$

$$\text{s.t.} \quad \sum_{j \in J} x_{ij} \leq s_i, \quad \forall i \in I \quad \text{(supply)}$$

$$\sum_{i \in I} x_{ij} \geq d_j, \quad \forall j \in J \quad \text{(demand)}$$

$$x_{ij} \geq 0$$

**Note:** When $\sum_i s_i = \sum_j d_j$ the problem is **balanced** and all constraints become equalities.

In [ ]:
import pyomo.environ as pyo
import json

print(f"Pyomo version: {pyo.__version__}")

## Loading the Instance

Our test instance: 3 suppliers, 4 customers, known optimal = **520**.

In [ ]:
with open("../instances/small_3x4.json") as f:
    data = json.load(f)

suppliers = data["suppliers"]
customers = data["customers"]
costs = data["costs"]
m = len(suppliers)
n = len(customers)

print(f"Suppliers: {m}  |  Customers: {n}")
print(f"Total supply: {sum(s['supply'] for s in suppliers)}")
print(f"Total demand: {sum(c['demand'] for c in customers)}")
print()

# Cost matrix
print("Cost matrix (suppliers x customers):")
header = "     " + "".join(f"C{j+1:>5}" for j in range(n))
print(header)
for i in range(m):
    row = f"S{i+1}   " + "".join(f"{costs[i][j]:>5}" for j in range(n))
    print(row)

## Building the Pyomo Model

In [ ]:
model = pyo.ConcreteModel(name="Transportation")

# ── Sets ──────────────────────────────────────────
model.I = pyo.Set(initialize=range(m), doc="Suppliers")
model.J = pyo.Set(initialize=range(n), doc="Customers")

# ── Decision Variables ────────────────────────────
model.x = pyo.Var(model.I, model.J, domain=pyo.NonNegativeReals,
                   doc="x[i,j] = quantity shipped from i to j")

# ── Objective: minimize total cost ────────────────
model.obj = pyo.Objective(
    expr=sum(costs[i][j] * model.x[i, j] for i in range(m) for j in range(n)),
    sense=pyo.minimize,
)

# ── Supply constraints ────────────────────────────
model.supply = pyo.Constraint(
    model.I,
    rule=lambda mdl, i: sum(mdl.x[i, j] for j in range(n)) <= suppliers[i]["supply"]
)

# ── Demand constraints ────────────────────────────
model.demand = pyo.Constraint(
    model.J,
    rule=lambda mdl, j: sum(mdl.x[i, j] for i in range(m)) >= customers[j]["demand"]
)

print("Model built!")
print(f"  Variables: {m * n}")
print(f"  Constraints: {m} (supply) + {n} (demand) = {m + n}")

## Solving

In [ ]:
solver = pyo.SolverFactory("appsi_highs")
result = solver.solve(model, tee=False)

print(f"Status    : {result.solver.termination_condition}")
print(f"Total cost: {pyo.value(model.obj):.2f}")
if data.get("optimal_value"):
    print(f"Known opt : {data['optimal_value']}")

## Analyzing the Shipping Plan

In [ ]:
print("Shipping Plan (quantity from supplier i to customer j):")
print()
header = "       " + "".join(f"C{j+1:>8}" for j in range(n)) + "  | Shipped  Supply"
print(header)
print("-" * len(header))

for i in range(m):
    shipped = 0
    row = f"S{i+1}     "
    for j in range(n):
        val = pyo.value(model.x[i, j])
        shipped += val
        row += f"{val:>8.1f}"
    cap = suppliers[i]["supply"]
    row += f"  | {shipped:>7.1f}  {cap:>6}"
    print(row)

print()
received = [sum(pyo.value(model.x[i, j]) for i in range(m)) for j in range(n)]
print("Received: " + "".join(f"{received[j]:>8.1f}" for j in range(n)))
print("Demand:   " + "".join(f"{customers[j]['demand']:>8}" for j in range(n)))

## Sensitivity Analysis — Shadow Prices

For LP problems, dual values (shadow prices) reveal how much the objective
would change per unit increase in the right-hand side of each constraint.

- **Supply shadow price $\pi_i$**: marginal value of one more unit of supply at supplier $i$
- **Demand shadow price $\mu_j$**: marginal cost of increasing demand at customer $j$ by one unit

In [ ]:
print("Supply shadow prices (dual values):")
for i in range(m):
    dual_val = model.dual[model.supply[i]]
    print(f"  Supplier {i+1}: {dual_val:>8.2f}")

print()
print("Demand shadow prices (dual values):")
for j in range(n):
    dual_val = model.dual[model.demand[j]]
    print(f"  Customer {j+1}: {dual_val:>8.2f}")

## Key Takeaways

1. The Transportation Problem is a **pure LP** — no integer variables needed.
   The constraint matrix has a special structure (totally unimodular)
   that guarantees integer solutions naturally.

2. **Shadow prices** help answer questions like:
   - "Should we expand a warehouse?" (look at supply dual)
   - "How much does serving a new customer cost at the margin?" (look at demand dual)

3. Extensions include:
   - **Unbalanced** problems (supply != demand)
   - **Transshipment** problems (intermediate nodes)
   - **Multi-commodity** flows

---

### References

- Hitchcock, F. L. (1941). The distribution of a product from several sources to numerous localities. *JMP*, 20, 224-230.
- Hillier, F. S., Lieberman, G. J. (2014). *Introduction to Operations Research* (10th ed.). McGraw-Hill. Ch. 8.
- Taha, H. A. (2017). *Operations Research: An Introduction* (10th ed.). Pearson. Ch. 5.